# HGSOC malignant-cell subtype baselines

See [README.md](README.md) for the execution order, upstream inputs, commands and paper mapping.
Run the unified entry point to save executed copies and local outputs. Scientific variants and their parameters are retained below.


## 1. Imports and device setup

In [ ]:
# SpiderNet utility imports also provide torch for device selection.
import json
from SpiderNet.utils import *
from SpiderNet.config import *
from dataclasses import fields
import scanpy as sc
import gseapy as gp

cuda_available = torch.cuda.is_available()
if cuda_available:
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")

## 2. Configure input and output locations

Set these paths to the matching HGSOC inputs and training run. When present, `HGSOC_modeltraining_setup.json` overrides the processed-data and output roots; the auxiliary-data root remains the value specified here.


In [ ]:
from workflow_paths import (PROCESSED_DATA_DIR, RESULTS_ROOT, DATA_ROOT, OUTPUT_ROOT, input_path, output_path, saved, ensure_output)
ensure_output()


## 3. Load the training-run manifest

Read `run_dirs.json` from the output root. The run-directory name must contain `Result_dim` followed by the MI dimensionality.


In [ ]:
import re
from workflow_paths import run_dirs as _configured_run_dirs
run_dirs = dict(_configured_run_dirs)
run_dir = Path(run_dirs["run_dir"])
match = re.search(r"Result_dim(\d+)", str(run_dir))
if match is None:
    raise ValueError(f"Cannot parse dim_envir from run directory: {run_dir}")
dim_envir = int(match.group(1))
print(run_dirs)


## 4. Load processed HGSOC inputs

The processed directory supplies AnnData, sample metadata, LR resources, and PyG objects. The next cell transfers every PyG object to the selected device, so the processed graph file is required even though these baseline analyses do not fit a SpiderNet model.


In [ ]:
from SpiderNet.io import load_processed_data, get_spidernet_pyg_list_path, load_spidernet_pyg_list, spidernet_pyg_list_exists
processed = load_processed_data(PROCESSED_DATA_DIR)

[f.name for f in fields(processed)]

In [ ]:
processed.spidernet_data = [data.to(device) for data in processed.spidernet_data]

## 5. Compare malignant-cell baseline representations

The malignant-cell AnnData is an upstream result. Its cell barcodes, sample annotations, expression layers, and existing score columns are retained for the baseline analyses.


### Load the malignant-cell subset


In [ ]:
cellclass_choose = 'Malignant'

In [ ]:
adata_choose_path = run_dirs['run_dir'] + "/" + ("adata_choose_" + str(cellclass_choose) + ".h5ad")
adata_choose = sc.read_h5ad(input_path(adata_choose_path))
print(adata_choose)

In [ ]:
# CancerSEA functional-state scores and embedding-analysis helpers
# Existing score columns are reused; missing columns are computed from adata_choose.X.
# The helper below can plot scores and calculate Moran's I on a chosen embedding.
# It is defined here but is not called by the remaining cells.

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

# =============================
# Figure style
# =============================
plt.close("all")
plt.style.use("default")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 8,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Functional-state score columns
# =============================
functional_states = [
    "Angiogenesis_score", "Apoptosis_score", "Cell Cycle_score",
    "Differentiation_score", "EMT_score", "Hypoxia_score",
    "Inflammation_score", "Metastasis_score",
    "Quiescence_score", "Stemness_score"
]

# =============================
# Compute CancerSEA functional-state scores on the original malignant-cell AnnData
# =============================
# Score before Scanpy's HVG subsetting. The plotting helper, when called,
# transfers these scores to a baseline AnnData by obs_names.
geneset_path = DATA_ROOT / "CancerSEA_OV" / "functional_geneset_list_df.csv"
if not input_path(geneset_path).exists():
    raise FileNotFoundError(f"Cannot find CancerSEA gene set file: {geneset_path}")

geneset_df = pd.read_csv(input_path(geneset_path))
geneset_df = geneset_df[geneset_df["Gene"].isin(adata_choose.var_names)].copy()

geneset_index_dict = {
    term: np.where(np.isin(adata_choose.var_names, geneset_df.loc[geneset_df["GeneSet"] == term, "Gene"]))[0]
    for term in np.unique(geneset_df["GeneSet"])
}

def _score_gene_set_max_normalized(adata, gene_indices):
    """
    Score one gene set by averaging gene-wise max-normalized expression.

    Sparse-safe implementation:
    - subset only the genes in the gene set
    - convert the small max-vector to dense before adding epsilon
    - avoid densifying the whole expression matrix
    """
    if len(gene_indices) == 0:
        return np.zeros(adata.n_obs, dtype=float)

    X_sub = adata.X[:, gene_indices]

    if sparse.issparse(X_sub):
        # scipy sparse max(axis=0) can return a sparse matrix/array.
        # Convert this 1 x n_genes vector to dense before scalar operations.
        gene_max = X_sub.max(axis=0)
        if sparse.issparse(gene_max):
            gene_max = gene_max.toarray()
        gene_max = np.asarray(gene_max, dtype=float).ravel()

        # Avoid division by zero for genes with all-zero expression.
        gene_max_safe = gene_max.copy()
        gene_max_safe[~np.isfinite(gene_max_safe)] = 0.0
        gene_max_safe[gene_max_safe <= 0] = 1e-8

        inv_gene_max = 1.0 / gene_max_safe
        score = np.asarray(X_sub.multiply(inv_gene_max).mean(axis=1)).ravel()
    else:
        X_sub = np.asarray(X_sub, dtype=float)
        gene_max = np.nanmax(X_sub, axis=0)

        gene_max_safe = gene_max.copy()
        gene_max_safe[~np.isfinite(gene_max_safe)] = 0.0
        gene_max_safe[gene_max_safe <= 0] = 1e-8

        score = np.nanmean(X_sub / gene_max_safe, axis=1)

    score = np.asarray(score, dtype=float).ravel()
    score[~np.isfinite(score)] = 0.0
    return score

for term, idxs in geneset_index_dict.items():
    score_col = f"{term}_score"
    if score_col not in adata_choose.obs.columns:
        adata_choose.obs[score_col] = _score_gene_set_max_normalized(adata_choose, idxs)

missing_states = [s for s in functional_states if s not in adata_choose.obs.columns]
if len(missing_states) > 0:
    raise KeyError(
        "The following functional-state score columns were not generated in adata_choose.obs: "
        f"{missing_states}. Check CancerSEA_OV/functional_geneset_list_df.csv."
    )

# Save shared scores once.
shared_score_out_dir = Path(run_dirs["run_dir"]) / "Baseline_FunctionalState_scores"
shared_score_out_dir.mkdir(parents=True, exist_ok=True)

score_df = adata_choose.obs[functional_states].copy()
score_df.insert(0, "cell_id", adata_choose.obs_names.astype(str))
score_df.to_csv(shared_score_out_dir / "CancerSEA_functional_state_scores_malignant_cells.csv", index=False)

print(f"Computed/reused functional-state scores for {len(functional_states)} states.")
print(f"Saved shared score table to: {shared_score_out_dir}")

# =============================
# Helper functions
# =============================







### Scanpy expression baseline

Compute the expression embedding and clusters, save and reload the baseline AnnData, plot annotations, and calculate sample-based ASW. The KEGG section also defines helpers used later by Banksy.


In [ ]:
import scanpy as sc

def run_logPCAUMAP(
    adata_choose,
    n_hvg=2000,
    n_pcs=50,
    n_neighbors=15,
    umap_min_dist=0.5,
    umap_spread=1.0,
    use_raw_counts=True,
    random_state=0
):
    """
    Return an HVG-subsetted copy with scaled expression, PCA, Louvain, and UMAP.

    If use_raw_counts is True, initialize X from the counts layer or adata.raw
    when available; otherwise retain the input X as the starting expression.
    Total-count normalization, log1p, HVG selection, and scaling always run.
    The returned object stores X_pca and X_umap in obsm and X_louvain in obs;
    the input AnnData is not modified.
    """

    adata = adata_choose.copy()

    # Select the starting expression matrix.
    if use_raw_counts:
        # Prefer the counts layer, then adata.raw, when available.
        if hasattr(adata, "layers") and ("counts" in adata.layers.keys()):
            adata.X = adata.layers["counts"].copy()
        elif adata.raw is not None:
            adata.X = adata.raw.X.copy()
        # Otherwise, the input X is assumed to contain counts.

    # ---------- 1) normalization + log ----------
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # Select HVGs from the normalized, log-transformed matrix used here.
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=n_hvg,
        flavor="seurat_v3"
    )
    adata = adata[:, adata.var["highly_variable"]].copy()

    # ---------- 3) scale ----------
    sc.pp.scale(adata, max_value=10)

    # ---------- 4) PCA ----------
    sc.tl.pca(adata, n_comps=n_pcs, svd_solver="arpack", random_state=random_state)

    # ---------- 5) neighbors ----------
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=min(n_pcs, adata.obsm["X_pca"].shape[1]))
    
    # Cluster the neighborhood graph before computing UMAP.
    sc.tl.louvain(adata, resolution=0.17, key_added='X_louvain')

    # ---------- 6) UMAP ----------
    sc.tl.umap(adata, min_dist=umap_min_dist, spread=umap_spread, random_state=random_state)

    return adata

adata_choose2 = run_logPCAUMAP(adata_choose)
adata_choose2.write_h5ad(run_dirs['run_dir'] + f"/adata_choose2_{cellclass_choose}.h5ad")

In [ ]:
run_dirs['run_dir']

In [ ]:
# Reload the saved Scanpy baseline AnnData.
adata_choose2 = sc.read_h5ad(input_path(run_dirs['run_dir'] + f"/adata_choose2_{cellclass_choose}.h5ad"))

In [ ]:
# Feature_show_umap uses the annotations implemented by SpiderNet.analysis.
# This helper plots stage_x and saves UMAP_stage_x.pdf.
from SpiderNet.analysis import Feature_show_umap
Feature_show_umap(
    cellclass_choose=cellclass_choose,
    adata_choose_path=input_path(run_dirs['run_dir'] + f"/adata_choose2_{cellclass_choose}.h5ad"),
    file_savepath_main=run_dirs['run_dir'],
    obsm_show="X_umap",
    adata_copy_path=input_path(str(PROCESSED_DATA_DIR / "adata_all.h5ad")),
    show=True
)

In [ ]:
from SpiderNet.analysis import plot_louvain_umap
plot_louvain_umap(
    adata_choose2,
    obsm_key="X_umap",       
    louvain_key="X_louvain", 
    out_dir=run_dirs['run_dir'],
    out_prefix=f"UMAP_{cellclass_choose}_Xlouvain",
    title=None,
    size=2,
    show=True
)


In [ ]:
# Sample-based average silhouette width in Scanpy UMAP space.
import numpy as np
from sklearn.metrics import silhouette_samples

# Use the two-dimensional embedding and the samples annotation.
X = adata_choose2.obsm['X_umap']
labels = adata_choose2.obs['samples'].values

# Draw 30% of cells, with a minimum of 1,000, using the existing seed.
np.random.seed(0)
n_cells = X.shape[0]
n_sub = max(int(n_cells * 0.30), 1000)
idx = np.random.choice(n_cells, size=n_sub, replace=False)

X_sub = X[idx]
labels_sub = labels[idx]
print(f"Computing silhouette score on {n_sub} subsampled cells (out of {n_cells})...")

# Average per-cell Euclidean silhouette values within the subsample.
sil_scores_sub = silhouette_samples(X_sub, labels_sub, metric='euclidean')
asw_sub = sil_scores_sub.mean()

print("Subsampled mean silhouette score:", asw_sub)

In [ ]:
# Scanpy baseline: continuous KEGG module scores on X_umap
# Load the four pathway gene sets listed below, select an expression source,
# and average gene-wise z-scores across matched malignant cells for each module.
# Save the scores, genes used, expression-source summary, plots, and AnnData.
# These helpers are also used by the Banksy module-score section.

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse
import scanpy as sc

# =============================
# Settings
# =============================
CLUSTER_COL = "X_louvain"

KEGG_PATHWAYS_TO_SCORE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

# =============================
# Figure style
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _safe_name(x):
    x = str(x)
    x = x.replace("/", "_").replace("-", "_")
    x = re.sub(r"[^\w]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _normalize_kegg_term(x):
    x = str(x).strip()
    x = re.sub(r"\s*\([^)]*\)\s*$", "", x)
    x = re.sub(r"\s+", " ", x)
    return x.lower()


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _first_indexer(values, query_values):
    mapping = {}

    for i, v in enumerate(values):
        v = str(v)
        if v not in mapping:
            mapping[v] = i

    return np.array(
        [mapping.get(str(q), -1) for q in query_values],
        dtype=int
    )


def _make_composite(sample_values, id_values):
    return np.array(
        [f"{str(s)}||{str(i)}" for s, i in zip(sample_values, id_values)],
        dtype=object
    )


def _add_expr_candidate(candidates, name, obj):
    if obj is not None and hasattr(obj, "var_names") and hasattr(obj, "obs") and hasattr(obj, "X"):
        candidates.append((name, obj))


def _build_expr_candidates(target_adata):
    candidates = []

    _add_expr_candidate(candidates, "target_adata", target_adata)

    if "adata_choose" in globals():
        _add_expr_candidate(candidates, "adata_choose", adata_choose)

    if "processed" in globals():
        _add_expr_candidate(candidates, "processed.adata_all", getattr(processed, "adata_all", None))
        _add_expr_candidate(candidates, "processed.adata", getattr(processed, "adata", None))

        if hasattr(processed, "adata_list") and len(processed.adata_list) > 0:
            try:
                import anndata as ad

                adata_list_concat = ad.concat(
                    processed.adata_list,
                    join="outer",
                    index_unique=None,
                    merge="same",
                )

                _add_expr_candidate(candidates, "concat(processed.adata_list)", adata_list_concat)

            except Exception as e:
                print(f"[Info] Could not concatenate processed.adata_list: {e}")

    for var_name in [
        "adata_all_full",
        "adata_raw",
        "adata_full",
        "adata_ori",
        "adata_original",
        "adata_copy",
        "adata",
        "adata_all",
    ]:
        if var_name in globals():
            _add_expr_candidate(candidates, var_name, globals()[var_name])

    seen = set()
    unique_candidates = []

    for name, obj in candidates:
        if id(obj) not in seen:
            unique_candidates.append((name, obj))
            seen.add(id(obj))

    return unique_candidates


def _match_target_rows_to_expr_adata(target_adata, expr_adata):
    if expr_adata is target_adata:
        return np.arange(target_adata.n_obs), "direct target_adata row order"

    target_obs_names = target_adata.obs_names.astype(str).to_numpy()

    if "barcode" in target_adata.obs.columns:
        target_barcodes = target_adata.obs["barcode"].astype(str).to_numpy()
    else:
        target_barcodes = target_obs_names.copy()

    expr_obs_names = expr_adata.obs_names.astype(str).to_numpy()

    if "barcode" in expr_adata.obs.columns:
        expr_barcodes = expr_adata.obs["barcode"].astype(str).to_numpy()
    else:
        expr_barcodes = expr_obs_names.copy()

    sample_col_target = _sample_col_from_obs(target_adata)
    sample_col_expr = _sample_col_from_obs(expr_adata)

    if sample_col_target is not None and sample_col_expr is not None:
        target_sample_values = target_adata.obs[sample_col_target].astype(str).to_numpy()
        expr_sample_values = expr_adata.obs[sample_col_expr].astype(str).to_numpy()

        query_comp_barcode = _make_composite(target_sample_values, target_barcodes)
        expr_comp_barcode = _make_composite(expr_sample_values, expr_barcodes)

        idx = _first_indexer(expr_comp_barcode, query_comp_barcode)
        if np.all(idx >= 0):
            return idx, f"sample + barcode using target['{sample_col_target}'] and expr['{sample_col_expr}']"

        query_comp_obs = _make_composite(target_sample_values, target_obs_names)
        expr_comp_obs = _make_composite(expr_sample_values, expr_obs_names)

        idx = _first_indexer(expr_comp_obs, query_comp_obs)
        if np.all(idx >= 0):
            return idx, f"sample + obs_names using target['{sample_col_target}'] and expr['{sample_col_expr}']"

    idx = _first_indexer(expr_barcodes, target_barcodes)
    if np.all(idx >= 0):
        return idx, "barcode -> expression barcode"

    idx = _first_indexer(expr_obs_names, target_obs_names)
    if np.all(idx >= 0):
        return idx, "target obs_names -> expression obs_names"

    idx = _first_indexer(expr_obs_names, target_barcodes)
    if np.all(idx >= 0):
        return idx, "target barcode -> expression obs_names"

    return None, None


def _get_X_array(adata, rows, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[rows, :][:, gene_idx]

    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub


def _load_kegg_genes(out_dir):
    """
    Load four selected KEGG pathway gene sets.

    Priority:
      1. global c5_kegg_reference_gene_df
      2. saved CSV
      3. fetch KEGG library with gseapy
    """
    if "c5_kegg_reference_gene_df" in globals():
        kegg_gene_df = c5_kegg_reference_gene_df.copy()

        required_cols = {"Requested_Pathway", "Gene"}
        missing_cols = required_cols - set(kegg_gene_df.columns)

        if len(missing_cols) == 0:
            print("Using KEGG pathway genes from global c5_kegg_reference_gene_df.")
            return kegg_gene_df

        print(
            "[Warning] c5_kegg_reference_gene_df exists but lacks required columns. "
            f"Missing: {missing_cols}. Will try saved files / gseapy."
        )

    candidate_paths = []

    if "run_dirs" in globals() and isinstance(run_dirs, dict) and "run_dir" in run_dirs:
        candidate_paths.append(
            Path(run_dirs["run_dir"])
            / "TCGA_OV_KEGG_gene_signature"
            / "C5_four_KEGG_pathway_reference_genes_long.csv"
        )

    candidate_paths.extend([
        Path(out_dir) / "TCGA_OV_KEGG_gene_signature" / "C5_four_KEGG_pathway_reference_genes_long.csv",
        (DATA_ROOT / 'TCGA_OV_KEGG_gene_signature/C5_four_KEGG_pathway_reference_genes_long.csv'),
        (DATA_ROOT / 'C5_four_KEGG_pathway_reference_genes_long.csv'),
    ])

    for candidate_path in candidate_paths:
        if input_path(candidate_path).exists():
            print(f"Loading KEGG pathway genes from saved CSV:\n{candidate_path}")

            kegg_gene_df = pd.read_csv(input_path(candidate_path))

            required_cols = {"Requested_Pathway", "Gene"}
            missing_cols = required_cols - set(kegg_gene_df.columns)

            if len(missing_cols) > 0:
                print(
                    f"[Warning] Found file but missing required columns {missing_cols}. "
                    f"Available columns: {list(kegg_gene_df.columns)}. Trying next source."
                )
                continue

            return kegg_gene_df

    print(
        "c5_kegg_reference_gene_df was not found, and no saved KEGG CSV was found. "
        "Fetching KEGG gene sets with gseapy..."
    )

    try:
        import gseapy as gp
        import difflib
    except ImportError as e:
        raise ImportError(
            "gseapy is required to fetch KEGG gene sets when the saved KEGG CSV is unavailable. "
            "Please install gseapy or run the KEGG export cell once in the main notebook."
        ) from e

    kegg_library_name = gene_set if "gene_set" in globals() else "KEGG_2021_Human"
    kegg_organism_label = organism_label if "organism_label" in globals() else "human"

    print(f"Loading KEGG reference library: {kegg_library_name} ({kegg_organism_label})")

    try:
        kegg_reference_library = gp.get_library(
            name=kegg_library_name,
            organism=kegg_organism_label,
        )
    except TypeError:
        kegg_reference_library = gp.get_library(name=kegg_library_name)

    available_terms = list(kegg_reference_library.keys())
    term_lookup = {term: term for term in available_terms}
    term_lookup.update({_normalize_kegg_term(term): term for term in available_terms})

    resolved_terms = {}
    missing_terms = []

    for term in KEGG_PATHWAYS_TO_SCORE:
        if term in kegg_reference_library:
            resolved_terms[term] = term
        elif _normalize_kegg_term(term) in term_lookup:
            resolved_terms[term] = term_lookup[_normalize_kegg_term(term)]
        else:
            missing_terms.append(term)

    if len(missing_terms) > 0:
        print("Could not find the following requested KEGG terms:")

        for term in missing_terms:
            close_matches = difflib.get_close_matches(
                term,
                available_terms,
                n=5,
                cutoff=0.35,
            )
            print(f"  - {term}")
            print(f"    closest matches: {close_matches}")

        raise KeyError("Some requested KEGG terms were not found in the KEGG reference library.")

    rows = []

    for requested_term, library_term in resolved_terms.items():
        reference_genes = sorted(set(map(str, kegg_reference_library[library_term])))

        for gene in reference_genes:
            rows.append({
                "Requested_Pathway": requested_term,
                "Library_Pathway": library_term,
                "Gene": gene,
                "Reference_Gene": gene,
            })

    kegg_gene_df = pd.DataFrame(rows).drop_duplicates()

    if kegg_gene_df.empty:
        raise ValueError("No KEGG genes were retrieved for the selected pathways.")

    save_dir = (
        Path(run_dirs["run_dir"]) / "TCGA_OV_KEGG_gene_signature"
        if "run_dirs" in globals() and "run_dir" in run_dirs
        else Path(out_dir) / "TCGA_OV_KEGG_gene_signature"
    )

    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / "C5_four_KEGG_pathway_reference_genes_long.csv"
    kegg_gene_df.to_csv(save_path, index=False)

    summary_df = (
        kegg_gene_df
        .groupby(["Requested_Pathway", "Library_Pathway"], as_index=False)
        .agg(n_reference_genes=("Gene", "nunique"))
    )

    summary_path = save_dir / "C5_four_KEGG_pathway_reference_gene_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"Saved KEGG pathway gene table to:\n{save_path}")
    print("KEGG pathway gene counts:")
    _display_df(summary_df)

    return kegg_gene_df


def _run_baseline_kegg_score_umap(
    target_adata,
    *,
    embedding_key,
    baseline_label,
    cluster_col=None,
    save_h5ad_path=None,
):
    if embedding_key not in target_adata.obsm:
        raise KeyError(
            f"Cannot find target_adata.obsm['{embedding_key}']. "
            f"Available obsm keys: {list(target_adata.obsm.keys())}"
        )

    coords = np.asarray(target_adata.obsm[embedding_key], dtype=float)

    if coords.ndim != 2 or coords.shape[1] < 2:
        raise ValueError(f"target_adata.obsm['{embedding_key}'] must be n_cells × 2 or higher.")

    coords = coords[:, :2]

    baseline_safe = _safe_name(baseline_label)

    base_out_dir = (
        Path(run_dirs["run_dir"])
        if "run_dirs" in globals() and "run_dir" in run_dirs
        else Path(".")
    )

    out_dir = base_out_dir / f"Baseline_{baseline_safe}_KEGG_module_score_continuous_UMAP"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Load KEGG gene sets
    # -----------------------------
    kegg_gene_df = _load_kegg_genes(out_dir=base_out_dir)

    required_cols = {"Requested_Pathway", "Gene"}
    missing_cols = required_cols - set(kegg_gene_df.columns)

    if len(missing_cols) > 0:
        raise KeyError(
            f"KEGG pathway-gene table is missing required columns: {missing_cols}. "
            f"Available columns: {list(kegg_gene_df.columns)}"
        )

    kegg_gene_df = (
        kegg_gene_df
        .loc[
            kegg_gene_df["Requested_Pathway"].isin(KEGG_PATHWAYS_TO_SCORE),
            ["Requested_Pathway", "Gene"]
        ]
        .dropna()
        .drop_duplicates()
        .copy()
    )

    if kegg_gene_df.empty:
        raise ValueError("No KEGG pathway genes found for selected four pathways.")

    geneset_dict = {}

    for pathway in KEGG_PATHWAYS_TO_SCORE:
        genes_cur = (
            kegg_gene_df
            .loc[kegg_gene_df["Requested_Pathway"] == pathway, "Gene"]
            .astype(str)
            .tolist()
        )

        genes_cur = list(dict.fromkeys([g.strip() for g in genes_cur if g.strip()]))

        score_name = f"KEGG_{_safe_name(pathway)}_score"
        geneset_dict[score_name] = genes_cur

    score_title_map = {
        f"KEGG_{_safe_name('HIF-1 signaling pathway')}_score": "HIF-1 signaling",
        f"KEGG_{_safe_name('PD-L1 expression and PD-1 checkpoint pathway in cancer')}_score": "PD-1/PD-L1 checkpoint",
        f"KEGG_{_safe_name('PI3K-Akt signaling pathway')}_score": "PI3K-Akt signaling",
        f"KEGG_{_safe_name('ECM-receptor interaction')}_score": "ECM-receptor interaction",
    }

    all_genes = []

    for genes in geneset_dict.values():
        all_genes.extend(genes)

    all_genes = list(dict.fromkeys([str(g).strip() for g in all_genes if str(g).strip()]))

    geneset_input_long = []

    for score_name, genes in geneset_dict.items():
        for gene in genes:
            geneset_input_long.append({
                "module": score_name,
                "module_label": score_title_map.get(score_name, score_name),
                "Gene": gene,
            })

    geneset_input_long = pd.DataFrame(geneset_input_long)

    geneset_input_long.to_csv(
        out_dir / f"{baseline_safe}_input_KEGG_genesets_long.csv",
        index=False,
    )

    print(f"[{baseline_label}] Input KEGG gene-set sizes:")
    _display_df(
        geneset_input_long
        .groupby(["module", "module_label"], as_index=False)
        .agg(n_input_genes=("Gene", "nunique"))
    )

    # -----------------------------
    # Prefer the first cell-matched source with the greatest pathway-gene coverage.
    # -----------------------------
    expr_candidates = _build_expr_candidates(target_adata)

    candidate_summary = []
    best = None

    for name, cand in expr_candidates:
        resolved_cur, missing_cur = _resolve_gene_names(cand, all_genes)
        n_genes_cur = len(resolved_cur)

        row_idx_cur, match_mode_cur = _match_target_rows_to_expr_adata(target_adata, cand)
        rows_ok = row_idx_cur is not None

        per_module_found = {}

        for score_name, genes in geneset_dict.items():
            per_module_found[score_name] = len([g for g in genes if g in resolved_cur])

        candidate_summary.append({
            "candidate": name,
            "n_obs": cand.n_obs,
            "n_vars": cand.n_vars,
            "n_total_KEGG_genes_found": n_genes_cur,
            "target_rows_matched": rows_ok,
            "match_mode": match_mode_cur if match_mode_cur is not None else "NA",
            **{f"n_found__{k}": v for k, v in per_module_found.items()},
            "first_10_found_genes": ", ".join(list(resolved_cur.keys())[:10]),
        })

        all_modules_have_gene = all(v > 0 for v in per_module_found.values())

        if n_genes_cur > 0 and rows_ok and all_modules_have_gene:
            if best is None or n_genes_cur > best["n_genes"]:
                best = {
                    "name": name,
                    "adata": cand,
                    "resolved": resolved_cur,
                    "missing": missing_cur,
                    "row_idx": row_idx_cur,
                    "match_mode": match_mode_cur,
                    "n_genes": n_genes_cur,
                    "per_module_found": per_module_found,
                }

    candidate_summary_df = pd.DataFrame(candidate_summary)

    candidate_summary_df.to_csv(
        out_dir / f"{baseline_safe}_expression_object_search_summary.csv",
        index=False,
    )

    print(f"[{baseline_label}] Expression object search summary:")
    _display_df(candidate_summary_df)

    if best is None:
        raise ValueError(
            f"[{baseline_label}] Cannot find an expression AnnData that contains at least one gene "
            "for every KEGG module and matches target cells. Check expression_object_search_summary."
        )

    expr_adata = best["adata"]
    expr_rows = best["row_idx"]
    resolved_genes = best["resolved"]
    missing_genes = best["missing"]

    available_genes = [g for g in all_genes if g in resolved_genes]
    actual_gene_names = [resolved_genes[g] for g in available_genes]

    print(f"[{baseline_label}] Selected expression source: {best['name']}")
    print(f"[{baseline_label}] Target row matching mode: {best['match_mode']}")
    print(f"[{baseline_label}] Total KEGG module genes found: {len(available_genes)} / {len(all_genes)}")

    if len(missing_genes) > 0:
        print(f"[{baseline_label}] [Warning] Missing KEGG module genes skipped: {missing_genes}")

    # -----------------------------
    # Extract expression and compute z-scores
    # -----------------------------
    X_sub = _get_X_array(
        adata=expr_adata,
        rows=expr_rows,
        genes_actual=actual_gene_names,
    )

    expr_gene_df = pd.DataFrame(
        X_sub,
        columns=available_genes,
        index=target_adata.obs_names,
    )

    gene_mean = expr_gene_df.mean(axis=0)
    # pandas uses sample standard deviation (ddof=1); constant genes become zero after fillna.
    gene_std = expr_gene_df.std(axis=0).replace(0, np.nan)

    expr_z = (expr_gene_df - gene_mean) / gene_std
    expr_z = expr_z.fillna(0)

    # -----------------------------
    # Compute KEGG module scores
    # -----------------------------
    if "barcode" in target_adata.obs.columns:
        barcode_values = target_adata.obs["barcode"].astype(str).to_numpy()
    else:
        barcode_values = target_adata.obs_names.astype(str).to_numpy()

    module_score_df = pd.DataFrame({
        "cell_id": target_adata.obs_names.astype(str),
        "barcode": barcode_values,
    }, index=target_adata.obs_names)

    if cluster_col is not None and cluster_col in target_adata.obs.columns:
        module_score_df["cluster"] = target_adata.obs[cluster_col].astype(str).to_numpy()

    module_score_df[f"{baseline_safe}_UMAP1"] = coords[:, 0]
    module_score_df[f"{baseline_safe}_UMAP2"] = coords[:, 1]

    module_gene_count_records = []

    for score_name, genes in geneset_dict.items():
        genes_cur = [g for g in genes if g in expr_z.columns]

        if len(genes_cur) == 0:
            print(f"[{baseline_label}] [Warning] No available genes for {score_name}. Skipping.")
            continue

        module_score_df[score_name] = expr_z[genes_cur].mean(axis=1).to_numpy()
        target_adata.obs[score_name] = module_score_df[score_name].to_numpy()

        module_gene_count_records.append({
            "module": score_name,
            "module_label": score_title_map.get(score_name, score_name),
            "n_genes_used": len(genes_cur),
            "genes_used": ", ".join(genes_cur),
        })

    module_gene_count_df = pd.DataFrame(module_gene_count_records)

    score_cols = [r["module"] for r in module_gene_count_records]

    if len(score_cols) == 0:
        raise ValueError(f"[{baseline_label}] No KEGG module scores were computed.")

    module_gene_count_df.to_csv(
        out_dir / f"{baseline_safe}_KEGG_module_gene_count.csv",
        index=False,
    )

    print(f"[{baseline_label}] KEGG module genes used:")
    _display_df(module_gene_count_df[["module_label", "n_genes_used", "genes_used"]])

    module_score_df.to_csv(
        out_dir / f"{baseline_safe}_celllevel_KEGG_module_scores.csv",
        index=False,
    )

    # -----------------------------
    # Plot continuous KEGG module scores on baseline UMAP
    # -----------------------------
    n_panels = len(score_cols)
    n_cols = 2
    n_rows = int(np.ceil(n_panels / n_cols))

    plot_summary_records = []

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.2 * n_cols, 2.9 * n_rows),
        squeeze=False,
    )

    for ax_i, score_name in enumerate(score_cols):
        ax = axes.flat[ax_i]

        score_values = module_score_df[score_name].to_numpy(dtype=float)
        finite_score = score_values[np.isfinite(score_values)]

        if len(finite_score) == 0:
            vmin, vmax = 0, 1
            score_mean = np.nan
            score_median = np.nan
        else:
            vmin, vmax = np.nanpercentile(finite_score, [1, 99])

            if vmin == vmax:
                vmin = np.nanmin(finite_score)
                vmax = np.nanmax(finite_score)

            if vmin == vmax:
                vmin = vmin - 1e-6
                vmax = vmax + 1e-6

            score_mean = float(np.nanmean(finite_score))
            score_median = float(np.nanmedian(finite_score))

        plot_summary_records.append({
            "baseline": baseline_label,
            "embedding_key": embedding_key,
            "module": score_name,
            "module_label": score_title_map.get(score_name, score_name),
            "n_finite_cells": int(len(finite_score)),
            "score_mean": score_mean,
            "score_median": score_median,
            "vmin_p1": vmin,
            "vmax_p99": vmax,
        })

        sc_plot = ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=score_values,
            cmap="viridis",
            s=0.8,
            linewidths=0,
            alpha=0.9,
            vmin=vmin,
            vmax=vmax,
            rasterized=True,
        )

        ax.set_title(
            score_title_map.get(score_name, score_name),
            fontsize=8,
        )

        ax.set_xlabel(f"{baseline_label} UMAP1")
        ax.set_ylabel(f"{baseline_label} UMAP2")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", direction="out", length=2)

        ax.set_xticks([])
        ax.set_yticks([])

        cbar = plt.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.03)
        cbar.set_label("Module score", fontsize=6)
        cbar.ax.tick_params(labelsize=5)

    for j in range(n_panels, n_rows * n_cols):
        axes.flat[j].axis("off")

    fig.suptitle(
        f"KEGG module scores on {baseline_label} UMAP",
        y=1.02,
        fontsize=10,
    )

    plt.tight_layout()

    plt.savefig(
        out_dir / f"{baseline_safe}_UMAP_continuous_KEGG_module_scores.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.savefig(
        out_dir / f"{baseline_safe}_UMAP_continuous_KEGG_module_scores.pdf",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.show()
    plt.close()

    # -----------------------------
    # Save outputs
    # -----------------------------
    plot_summary_df = pd.DataFrame(plot_summary_records)

    plot_summary_df.to_csv(
        out_dir / f"{baseline_safe}_UMAP_continuous_KEGG_score_summary.csv",
        index=False,
    )

    module_score_df.to_csv(
        out_dir / f"{baseline_safe}_celllevel_KEGG_module_scores_with_coordinates.csv",
        index=False,
    )

    print(f"[{baseline_label}] Continuous KEGG module-score UMAP summary:")
    _display_df(plot_summary_df)

    if save_h5ad_path is not None:
        target_adata.write_h5ad(save_h5ad_path)

    return {
        "target_adata": target_adata,
        "module_score_df": module_score_df,
        "module_gene_count_df": module_gene_count_df,
        "plot_summary_df": plot_summary_df,
        "out_dir": out_dir,
    }


# ==============================================================
# Run Scanpy baseline analysis
# ==============================================================

if "adata_choose2" not in globals():
    adata_choose2 = sc.read_h5ad(input_path(run_dirs["run_dir"] + f"/adata_choose2_{cellclass_choose}.h5ad"))

scanpy_kegg_score_umap_result = _run_baseline_kegg_score_umap(
    target_adata=adata_choose2,
    embedding_key="X_umap",
    baseline_label="Scanpy",
    cluster_col="X_louvain",
    save_h5ad_path=run_dirs["run_dir"] + f"/adata_choose2_{cellclass_choose}.h5ad",
)

### Banksy baseline from precomputed results

Load the cell-identity matrix and the separate UMAP/cluster exports. The cell-identity matrix is displayed but is not consumed by subsequent analysis. UMAP coordinates and clusters are aligned by barcode before plotting, ASW calculation, and KEGG scoring.


In [ ]:
banksy_umap_path = DATA_ROOT / "Banksy/" f"Banksy_Umap_{cellclass_choose}.csv"
banksy_cluster_df_path = DATA_ROOT / "Banksy/" f"Banksy_cluster_{cellclass_choose}.csv"
banksy_umap_df = pd.read_csv(input_path(banksy_umap_path), index_col=0)
banksy_cluster_df = pd.read_csv(input_path(banksy_cluster_df_path), index_col=0)
adata_choose3 = adata_choose.copy()
# Align Banksy exports to the malignant-cell barcode order.
cells = adata_choose3.obs['barcode'].tolist()

umap_cols = banksy_umap_df.columns[:2]
banksy_umap_df_use = banksy_umap_df.loc[:, umap_cols].copy()
banksy_umap_df_use.index = banksy_umap_df_use.index.astype(str)

cluster_col = banksy_cluster_df.columns[0]
banksy_cluster_s = banksy_cluster_df[cluster_col].copy()
banksy_cluster_s.index = banksy_cluster_s.index.astype(str)
# Reindex without dropping unmatched cells; report missing entries below.
umap_aligned = banksy_umap_df_use.reindex(cells)
cluster_aligned = banksy_cluster_s.reindex(cells)

n_missing_umap = umap_aligned.isna().any(axis=1).sum()
n_missing_cluster = cluster_aligned.isna().sum()
print(f"Missing UMAP rows after align: {n_missing_umap}/{len(cells)}")
print(f"Missing cluster after align: {n_missing_cluster}/{len(cells)}")
# Attach the aligned embedding and categorical clusters.
adata_choose3.obsm["Banksy_umap"] = umap_aligned.values.astype(float)
adata_choose3.obs["Banksy_louvain"] = np.array(cluster_aligned.astype("category"))
adata_choose3.obs["Banksy_louvain"] = adata_choose3.obs["Banksy_louvain"].astype("category")
# Save the Banksy baseline AnnData.
adata_choose3.write_h5ad(run_dirs['run_dir'] + f"/adata_choose3_{cellclass_choose}.h5ad")

In [ ]:
# Feature_show_umap uses the annotations implemented by SpiderNet.analysis.
# This helper plots stage_x and saves UMAP_stage_x.pdf.
from SpiderNet.analysis import Feature_show_umap
Feature_show_umap(
    cellclass_choose=cellclass_choose,
    adata_choose_path=input_path(run_dirs['run_dir'] + f"/adata_choose3_{cellclass_choose}.h5ad"),
    file_savepath_main=run_dirs['run_dir'],
    obsm_show="Banksy_umap",
    adata_copy_path=input_path(str(PROCESSED_DATA_DIR / "adata_all.h5ad")),
    show=True
)

In [ ]:
from SpiderNet.analysis import plot_louvain_umap
plot_louvain_umap(
    adata_choose3,
    obsm_key="Banksy_umap",       
    louvain_key="Banksy_louvain", 
    out_dir=run_dirs['run_dir'],
    out_prefix=f"UMAP_{cellclass_choose}_BanksyCluster",
    title=None,
    size=2,
    show=True
)


In [ ]:
# Sample-based average silhouette width in Banksy UMAP space.
import numpy as np
from sklearn.metrics import silhouette_samples

# Use the two-dimensional embedding and the samples annotation.
X = adata_choose3.obsm['Banksy_umap']
labels = adata_choose3.obs['samples'].values

# Draw 30% of cells, with a minimum of 1,000, using the existing seed.
np.random.seed(0)
n_cells = X.shape[0]
n_sub = max(int(n_cells * 0.30), 1000)
idx = np.random.choice(n_cells, size=n_sub, replace=False)

X_sub = X[idx]
labels_sub = labels[idx]
print(f"Computing silhouette score on {n_sub} subsampled cells (out of {n_cells})...")

# Average per-cell Euclidean silhouette values within the subsample.
sil_scores_sub = silhouette_samples(X_sub, labels_sub, metric='euclidean')
asw_sub = sil_scores_sub.mean()

print("Subsampled mean silhouette score:", asw_sub)

In [ ]:
# Banksy baseline: continuous KEGG module scores on Banksy_umap
# Reuse the four pathways and scoring helpers defined in the Scanpy section.
# Save the scores, genes used, expression-source summary, plots, and AnnData.

if "_run_baseline_kegg_score_umap" not in globals():
    raise NameError(
        "Please run the Scanpy baseline KEGG module-score UMAP cell first, "
        "because it defines the shared helper function "
        "_run_baseline_kegg_score_umap()."
    )

if "adata_choose3" not in globals():
    adata_choose3 = sc.read_h5ad(
        input_path(run_dirs["run_dir"] + f"/adata_choose3_{cellclass_choose}.h5ad")
    )

banksy_kegg_score_umap_result = _run_baseline_kegg_score_umap(
    target_adata=adata_choose3,
    embedding_key="Banksy_umap",
    baseline_label="Banksy",
    cluster_col="Banksy_louvain",
    save_h5ad_path=run_dirs["run_dir"] + f"/adata_choose3_{cellclass_choose}.h5ad",
)

In [ ]:
# ==============================================================
# Banksy clusters: KEGG module-score heatmap + sample representation
# --------------------------------------------------------------
# For each Banksy_louvain cluster, this cell shows:
#   1. mean cell-level module score for the four KEGG pathways
#   2. number of samples where the cluster exceeds 1% of malignant cells
# ==============================================================

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
import scanpy as sc

# -----------------------------
# Settings
# -----------------------------
BANKSY_CLUSTER_COL = "Banksy_louvain"
SAMPLE_CLUSTER_MIN_PROP = 0.01

KEGG_PATHWAYS_TO_COMPARE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

OUT_PREFIX_BANKSY_HEATMAP = "Banksy_louvain_KEGG_module_score_heatmap_sample_representation"

out_dir = Path(run_dirs["run_dir"]) if "run_dirs" in globals() and "run_dir" in run_dirs else Path(".")
out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Plot style
# -----------------------------
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# -----------------------------
# Helpers
# -----------------------------
def _safe_name_banksy_heatmap(x):
    x = str(x)
    x = x.replace("/", "_").replace("-", "_")
    x = re.sub(r"[^\w]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _sample_col_from_obs_banksy_heatmap(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _display_df_banksy_heatmap(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _sort_cluster_values(values):
    values = [str(v) for v in pd.unique(values)]
    def _key(x):
        x2 = re.sub(r"^C", "", str(x), flags=re.IGNORECASE)
        return (0, int(x2)) if x2.isdigit() else (1, str(x))
    return sorted(values, key=_key)

# -----------------------------
# Load / validate Banksy AnnData
# -----------------------------
if "adata_choose3" not in globals():
    adata_choose3 = sc.read_h5ad(
        input_path(run_dirs["run_dir"] + f"/adata_choose3_{cellclass_choose}.h5ad")
    )

if BANKSY_CLUSTER_COL not in adata_choose3.obs.columns:
    raise KeyError(
        f"Cannot find '{BANKSY_CLUSTER_COL}' in adata_choose3.obs. "
        f"Available columns: {list(adata_choose3.obs.columns)}"
    )

expected_score_cols = [
    f"KEGG_{_safe_name_banksy_heatmap(pathway)}_score"
    for pathway in KEGG_PATHWAYS_TO_COMPARE
]

score_label_map = {
    f"KEGG_{_safe_name_banksy_heatmap('HIF-1 signaling pathway')}_score": "HIF-1",
    f"KEGG_{_safe_name_banksy_heatmap('PD-L1 expression and PD-1 checkpoint pathway in cancer')}_score": "PD-1/PD-L1",
    f"KEGG_{_safe_name_banksy_heatmap('PI3K-Akt signaling pathway')}_score": "PI3K-Akt",
    f"KEGG_{_safe_name_banksy_heatmap('ECM-receptor interaction')}_score": "ECM-receptor",
}

missing_score_cols = [c for c in expected_score_cols if c not in adata_choose3.obs.columns]
if len(missing_score_cols) > 0:
    raise KeyError(
        "The following KEGG module-score columns are missing from adata_choose3.obs. "
        "Please run the Banksy KEGG module-score UMAP cell first:\n"
        + "\n".join(missing_score_cols)
    )

sample_col = _sample_col_from_obs_banksy_heatmap(adata_choose3)
if sample_col is None:
    print("[Warning] No sample column found. Using one pseudo-sample '__all_cells__'.")
    sample_values = np.array(["__all_cells__"] * adata_choose3.n_obs, dtype=object)
    sample_col = "__all_cells__"
else:
    sample_values = adata_choose3.obs[sample_col].astype(str).to_numpy()

obs_df = pd.DataFrame({
    "cell_id": adata_choose3.obs_names.astype(str),
    "sample": sample_values,
    BANKSY_CLUSTER_COL: adata_choose3.obs[BANKSY_CLUSTER_COL].astype(str).to_numpy(),
})

for col in expected_score_cols:
    obs_df[col] = pd.to_numeric(adata_choose3.obs[col], errors="coerce").to_numpy()

cluster_order = _sort_cluster_values(obs_df[BANKSY_CLUSTER_COL])

# -----------------------------
# Mean module score by Banksy cluster
# -----------------------------
cluster_mean_df = (
    obs_df
    .groupby(BANKSY_CLUSTER_COL, observed=True)[expected_score_cols]
    .mean()
    .reindex(cluster_order)
)

cluster_size_df = (
    obs_df
    .groupby(BANKSY_CLUSTER_COL, observed=True)
    .size()
    .reindex(cluster_order)
    .rename("n_cells")
    .reset_index()
)

# -----------------------------
# Count samples where this cluster is >1% of the malignant-cell subset.
# -----------------------------
sample_cluster_count = (
    obs_df
    .groupby(["sample", BANKSY_CLUSTER_COL], observed=True)
    .size()
    .rename("n_cluster_cells")
    .reset_index()
)

sample_total = (
    obs_df
    .groupby("sample", observed=True)
    .size()
    .rename("n_sample_cells")
    .reset_index()
)

sample_cluster_prop = sample_cluster_count.merge(sample_total, on="sample", how="left")
sample_cluster_prop["cluster_prop_in_sample"] = (
    sample_cluster_prop["n_cluster_cells"] / sample_cluster_prop["n_sample_cells"]
)

sample_count_gt1_df = (
    sample_cluster_prop
    .loc[sample_cluster_prop["cluster_prop_in_sample"] > SAMPLE_CLUSTER_MIN_PROP]
    .groupby(BANKSY_CLUSTER_COL, observed=True)["sample"]
    .nunique()
    .reindex(cluster_order)
    .fillna(0)
    .astype(int)
    .rename("n_samples_cluster_gt1pct")
    .reset_index()
)

heatmap_summary_df = (
    cluster_size_df
    .merge(sample_count_gt1_df, on=BANKSY_CLUSTER_COL, how="left")
)
heatmap_summary_df["n_samples_cluster_gt1pct"] = heatmap_summary_df["n_samples_cluster_gt1pct"].fillna(0).astype(int)

# Add mean scores into one summary table
cluster_mean_reset = cluster_mean_df.reset_index()
heatmap_summary_df = heatmap_summary_df.merge(cluster_mean_reset, on=BANKSY_CLUSTER_COL, how="left")

heatmap_summary_df["mean_four_module_score"] = heatmap_summary_df[expected_score_cols].mean(axis=1)

# Save summary and sample-cluster proportion table
heatmap_summary_path = out_dir / f"{OUT_PREFIX_BANKSY_HEATMAP}_summary.csv"
sample_cluster_prop_path = out_dir / f"{OUT_PREFIX_BANKSY_HEATMAP}_sample_cluster_proportions.csv"
heatmap_summary_df.to_csv(heatmap_summary_path, index=False)
sample_cluster_prop.to_csv(sample_cluster_prop_path, index=False)

# -----------------------------
# Plot heatmap
# -----------------------------
heatmap_mat = cluster_mean_df.copy()
heatmap_mat.columns = [score_label_map[c] for c in heatmap_mat.columns]

row_label_map = heatmap_summary_df.set_index(BANKSY_CLUSTER_COL).to_dict(orient="index")
row_labels = []
for cl in cluster_order:
    rec = row_label_map[str(cl)]
    row_labels.append(
        f"{cl} | n={int(rec['n_cells'])}; samples>1%={int(rec['n_samples_cluster_gt1pct'])}"
    )

fig_height = max(2.8, 0.32 * len(cluster_order) + 1.2)
fig_width = 5.2

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

sns.heatmap(
    heatmap_mat,
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Mean module score"},
    yticklabels=row_labels,
    ax=ax,
)

ax.set_xlabel("KEGG module")
ax.set_ylabel("Banksy_louvain cluster | cells; sample representation")
ax.set_title("Banksy clusters: mean KEGG module score", fontsize=9)

ax.tick_params(axis="x", rotation=35)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")

plt.tight_layout()

plt.savefig(
    out_dir / f"{OUT_PREFIX_BANKSY_HEATMAP}.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.savefig(
    out_dir / f"{OUT_PREFIX_BANKSY_HEATMAP}.pdf",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()

print(f"Saved Banksy heatmap summary to:\n{heatmap_summary_path}")
print(f"Saved sample-cluster proportions to:\n{sample_cluster_prop_path}")
print("Banksy cluster KEGG module-score summary:")
_display_df_banksy_heatmap(heatmap_summary_df)


In [ ]:
# Compare SpiderNet malignant C5 with the fixed Banksy cluster 10
# Read the SpiderNet KEGG score export from HGSOC_Malignantsubtype_analysis_V2.ipynb
# and the Banksy scores computed above. Export cell-level and summary tables,
# then plot the four pathway-score distributions. No statistical test is run.

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
import scanpy as sc

# -----------------------------
# Settings
# -----------------------------
SPIDERNET_C5_LABEL = "5"
BANKSY_CLUSTER_COL = "Banksy_louvain"
BANKSY_CLUSTER_TO_COMPARE = "10"

KEGG_PATHWAYS_TO_COMPARE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

SPIDERNET_SCORE_CSV_PATH = (
    Path(run_dirs["run_dir"])
    / "HGSOC_malignant_cell_KEGG_module_scores_SpiderNet_MI_louvain.csv"
)

OUT_PREFIX_C5_VS_BANKSY = "SpiderNet_C5_vs_Banksy_C10_KEGG_module_score_boxplot"

out_dir = Path(run_dirs["run_dir"]) if "run_dirs" in globals() and "run_dir" in run_dirs else Path(".")
out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Plot style
# -----------------------------
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# -----------------------------
# Helpers
# -----------------------------
def _safe_name_c5_banksy(x):
    x = str(x)
    x = x.replace("/", "_").replace("-", "_")
    x = re.sub(r"[^\w]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _clean_cluster_label_c5_banksy(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _sample_col_from_obs_c5_banksy(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _display_df_c5_banksy(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


# -----------------------------
# KEGG score columns
# -----------------------------
score_cols = [
    f"KEGG_{_safe_name_c5_banksy(pathway)}_score"
    for pathway in KEGG_PATHWAYS_TO_COMPARE
]

score_label_map = {
    f"KEGG_{_safe_name_c5_banksy('HIF-1 signaling pathway')}_score": "HIF-1",
    f"KEGG_{_safe_name_c5_banksy('PD-L1 expression and PD-1 checkpoint pathway in cancer')}_score": "PD-1/PD-L1",
    f"KEGG_{_safe_name_c5_banksy('PI3K-Akt signaling pathway')}_score": "PI3K-Akt",
    f"KEGG_{_safe_name_c5_banksy('ECM-receptor interaction')}_score": "ECM-receptor",
}

# -----------------------------
# Read SpiderNet exported CSV
# -----------------------------
if not input_path(SPIDERNET_SCORE_CSV_PATH).exists():
    candidates = sorted(out_dir.glob("*malignant_cell*KEGG*SpiderNet*MI_louvain*.csv"))
    if len(candidates) > 0:
        SPIDERNET_SCORE_CSV_PATH = candidates[0]
        print(f"Using SpiderNet KEGG score CSV found by glob:\n{SPIDERNET_SCORE_CSV_PATH}")
    else:
        raise FileNotFoundError(
            "Cannot find SpiderNet malignant-cell KEGG score CSV. "
            "Please run the export cell in HGSOC_Malignantsubtype_analysis first.\n"
            f"Expected path:\n{SPIDERNET_SCORE_CSV_PATH}"
        )

spider_df = pd.read_csv(input_path(SPIDERNET_SCORE_CSV_PATH))

missing_spider_cols = [c for c in ["barcode", "sample"] + score_cols if c not in spider_df.columns]
if len(missing_spider_cols) > 0:
    raise KeyError(
        "SpiderNet score CSV is missing required columns:\n"
        + "\n".join(missing_spider_cols)
        + f"\nAvailable columns: {list(spider_df.columns)}"
    )

if "MI_louvain_clean" in spider_df.columns:
    spider_cluster_clean = spider_df["MI_louvain_clean"].astype(str).map(_clean_cluster_label_c5_banksy)
elif "MI_louvain" in spider_df.columns:
    spider_cluster_clean = spider_df["MI_louvain"].astype(str).map(_clean_cluster_label_c5_banksy)
elif "MalignantCluster" in spider_df.columns:
    spider_cluster_clean = (
        spider_df["MalignantCluster"]
        .astype(str)
        .str.extract(r"C([^\s]+)", expand=False)
        .fillna(spider_df["MalignantCluster"].astype(str))
    )
    spider_cluster_clean = spider_cluster_clean.map(_clean_cluster_label_c5_banksy)
else:
    raise KeyError("Cannot identify SpiderNet cluster column in the exported CSV.")

spider_df["MI_louvain_clean"] = spider_cluster_clean.astype(str)

spider_c5_df = spider_df.loc[
    spider_df["MI_louvain_clean"].astype(str) == str(SPIDERNET_C5_LABEL)
].copy()

if spider_c5_df.empty:
    raise ValueError(
        f"No SpiderNet malignant cells found for C{SPIDERNET_C5_LABEL}. "
        "Check MI_louvain_clean values in the exported CSV."
    )

# -----------------------------
# Load Banksy adata
# -----------------------------
if "adata_choose3" not in globals():
    adata_choose3 = sc.read_h5ad(
        input_path(run_dirs["run_dir"] + f"/adata_choose3_{cellclass_choose}.h5ad")
    )

if BANKSY_CLUSTER_COL not in adata_choose3.obs.columns:
    raise KeyError(
        f"Cannot find '{BANKSY_CLUSTER_COL}' in adata_choose3.obs. "
        f"Available columns: {list(adata_choose3.obs.columns)}"
    )

missing_banksy_scores = [c for c in score_cols if c not in adata_choose3.obs.columns]
if len(missing_banksy_scores) > 0:
    raise KeyError(
        "The following KEGG module-score columns are missing from adata_choose3.obs. "
        "Please run the Banksy KEGG module-score UMAP cell first:\n"
        + "\n".join(missing_banksy_scores)
    )

if "barcode" in adata_choose3.obs.columns:
    banksy_barcode = adata_choose3.obs["barcode"].astype(str).to_numpy()
else:
    banksy_barcode = adata_choose3.obs_names.astype(str).to_numpy()

banksy_sample_col = _sample_col_from_obs_c5_banksy(adata_choose3)
if banksy_sample_col is not None:
    banksy_sample = adata_choose3.obs[banksy_sample_col].astype(str).to_numpy()
else:
    banksy_sample = np.array(["__unknown_sample__"] * adata_choose3.n_obs, dtype=object)

banksy_cluster_raw = adata_choose3.obs[BANKSY_CLUSTER_COL].astype(str).to_numpy()
banksy_cluster_clean = np.array(
    [_clean_cluster_label_c5_banksy(x) for x in banksy_cluster_raw],
    dtype=object,
)

banksy_df = pd.DataFrame({
    "cell_id": adata_choose3.obs_names.astype(str),
    "barcode": banksy_barcode,
    "sample": banksy_sample,
    BANKSY_CLUSTER_COL: banksy_cluster_raw,
    f"{BANKSY_CLUSTER_COL}_clean": banksy_cluster_clean,
})

for col in score_cols:
    banksy_df[col] = pd.to_numeric(adata_choose3.obs[col], errors="coerce").to_numpy()

# -----------------------------
# Select Banksy cluster 10
# -----------------------------
banksy_cluster_mean_df = (
    banksy_df
    .groupby(f"{BANKSY_CLUSTER_COL}_clean", observed=True)[score_cols]
    .mean()
)

banksy_cluster_mean_df["mean_four_module_score"] = banksy_cluster_mean_df[score_cols].mean(axis=1)

BANKSY_CLUSTER_TO_COMPARE_RESOLVED = _clean_cluster_label_c5_banksy(BANKSY_CLUSTER_TO_COMPARE)
print(f"Using fixed Banksy_louvain cluster: {BANKSY_CLUSTER_TO_COMPARE_RESOLVED}")

banksy_selected_df = banksy_df.loc[
    banksy_df[f"{BANKSY_CLUSTER_COL}_clean"].astype(str) == BANKSY_CLUSTER_TO_COMPARE_RESOLVED
].copy()

if banksy_selected_df.empty:
    available_clusters = sorted(banksy_df[f"{BANKSY_CLUSTER_COL}_clean"].astype(str).unique())
    raise ValueError(
        f"No cells found for Banksy_louvain={BANKSY_CLUSTER_TO_COMPARE_RESOLVED}.\n"
        f"Available Banksy_louvain clusters are:\n{available_clusters}"
    )

# -----------------------------
# Add explicit labels
# -----------------------------
spider_method_group_label = f"SpiderNet\nMI-louvain C{SPIDERNET_C5_LABEL}"
banksy_method_group_label = f"Banksy\nlouvain C{BANKSY_CLUSTER_TO_COMPARE_RESOLVED}"

spider_c5_df["method"] = "SpiderNet"
spider_c5_df["cluster_label"] = f"MI-louvain C{SPIDERNET_C5_LABEL}"
spider_c5_df["method_group_label"] = spider_method_group_label
spider_c5_df["comparison_group"] = f"SpiderNet C{SPIDERNET_C5_LABEL}"
spider_c5_df["method_cluster"] = f"SpiderNet MI-louvain C{SPIDERNET_C5_LABEL}"

banksy_selected_df["method"] = "Banksy"
banksy_selected_df["cluster_label"] = f"Banksy-louvain C{BANKSY_CLUSTER_TO_COMPARE_RESOLVED}"
banksy_selected_df["method_group_label"] = banksy_method_group_label
banksy_selected_df["comparison_group"] = f"Banksy C{BANKSY_CLUSTER_TO_COMPARE_RESOLVED}"
banksy_selected_df["method_cluster"] = f"Banksy_louvain {BANKSY_CLUSTER_TO_COMPARE_RESOLVED}"

# -----------------------------
# Combine tables
# -----------------------------
common_cols = [
    "barcode",
    "sample",
    "method",
    "cluster_label",
    "method_group_label",
    "comparison_group",
    "method_cluster",
] + score_cols

comparison_df = pd.concat(
    [
        spider_c5_df[common_cols],
        banksy_selected_df[common_cols],
    ],
    axis=0,
    ignore_index=True,
)

comparison_long_df = comparison_df.melt(
    id_vars=[
        "barcode",
        "sample",
        "method",
        "cluster_label",
        "method_group_label",
        "comparison_group",
        "method_cluster",
    ],
    value_vars=score_cols,
    var_name="module",
    value_name="module_score",
)

comparison_long_df["module_label"] = comparison_long_df["module"].map(score_label_map)

comparison_summary_df = (
    comparison_long_df
    .groupby(
        [
            "method",
            "cluster_label",
            "method_group_label",
            "comparison_group",
            "method_cluster",
            "module",
            "module_label",
        ],
        observed=True,
    )
    .agg(
        n_cells=("module_score", "count"),
        n_samples=("sample", "nunique"),
        mean_score=("module_score", "mean"),
        median_score=("module_score", "median"),
        std_score=("module_score", "std"),
    )
    .reset_index()
)

banksy_cluster_mean_path = out_dir / f"{OUT_PREFIX_C5_VS_BANKSY}_Banksy_cluster_mean_scores.csv"
comparison_table_path = out_dir / f"{OUT_PREFIX_C5_VS_BANKSY}_celllevel_table.csv"
comparison_long_path = out_dir / f"{OUT_PREFIX_C5_VS_BANKSY}_celllevel_long.csv"
comparison_summary_path = out_dir / f"{OUT_PREFIX_C5_VS_BANKSY}_summary.csv"

banksy_cluster_mean_df.reset_index().to_csv(banksy_cluster_mean_path, index=False)
comparison_df.to_csv(comparison_table_path, index=False)
comparison_long_df.to_csv(comparison_long_path, index=False)
comparison_summary_df.to_csv(comparison_summary_path, index=False)

# -----------------------------
# Plot
# -----------------------------
module_order = [score_label_map[c] for c in score_cols]
group_order = [spider_method_group_label, banksy_method_group_label]

comparison_long_df["module_label"] = pd.Categorical(
    comparison_long_df["module_label"],
    categories=module_order,
    ordered=True,
)
comparison_long_df["method_group_label"] = pd.Categorical(
    comparison_long_df["method_group_label"],
    categories=group_order,
    ordered=True,
)

palette = {
    spider_method_group_label: "#D95F5F",
    banksy_method_group_label: "#6BAED6",
}

g = sns.catplot(
    data=comparison_long_df,
    x="method_group_label",
    y="module_score",
    col="module_label",
    col_order=module_order,
    order=group_order,
    kind="box",
    showfliers=False,
    linewidth=0.8,
    width=0.58,
    palette=palette,
    sharey=False,
    col_wrap=2,
    height=2.7,
    aspect=1.0,
)

for ax, module_label in zip(g.axes.flat, module_order):
    sub = comparison_long_df.loc[
        comparison_long_df["module_label"].astype(str) == str(module_label)
    ].copy()

    n_map = (
        sub
        .groupby("method_group_label", observed=True)["barcode"]
        .count()
        .to_dict()
    )

    y_min = sub["module_score"].min()
    y_max = sub["module_score"].max()
    y_range = y_max - y_min if y_max > y_min else 1.0

    for i, group in enumerate(group_order):
        n_cur = int(n_map.get(group, 0))
        ax.text(
            i,
            y_max + 0.06 * y_range,
            f"n={n_cur:,}",
            ha="center",
            va="bottom",
            fontsize=5.5,
            rotation=90,
        )

    ax.set_title(str(module_label), fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("Module score")

    ax.set_ylim(
        y_min - 0.08 * y_range,
        y_max + 0.22 * y_range,
    )

    # Display the method and cluster labels for both comparison groups.
    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(
        group_order,
        rotation=0,
        ha="center",
        va="top",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

g.fig.suptitle(
    f"SpiderNet MI-louvain C{SPIDERNET_C5_LABEL} vs Banksy-louvain C{BANKSY_CLUSTER_TO_COMPARE_RESOLVED}",
    y=0.98,
    fontsize=9,
)

# Reserve space for the multiline x-axis labels with explicit subplot margins.
g.fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.18,
    top=0.86,
    wspace=0.28,
    hspace=0.35,
)

plt.savefig(
    out_dir / f"{OUT_PREFIX_C5_VS_BANKSY}_BanksyC{BANKSY_CLUSTER_TO_COMPARE_RESOLVED}_method_labeled.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    out_dir / f"{OUT_PREFIX_C5_VS_BANKSY}_BanksyC{BANKSY_CLUSTER_TO_COMPARE_RESOLVED}_method_labeled.pdf",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()

print(f"SpiderNet score CSV used:\n{SPIDERNET_SCORE_CSV_PATH}")
print(f"Banksy cluster selected: {BANKSY_CLUSTER_TO_COMPARE_RESOLVED}")
print(f"Saved Banksy cluster mean score table to:\n{banksy_cluster_mean_path}")
print(f"Saved comparison table to:\n{comparison_table_path}")
print(f"Saved comparison long table to:\n{comparison_long_path}")
print(f"Saved comparison summary to:\n{comparison_summary_path}")
print("Comparison summary:")
_display_df_c5_banksy(comparison_summary_df)